# Tests du projet sur Google Colab

1. Clone du dépôt GitHub
2. Installation des dépendances de test (`requirements-test.txt`)
3. `pytest` + lint ruff
4. *(optionnel)* `dvc pull` depuis Google Drive, puis test de l'API avec le vrai modèle

Les tests unitaires utilisent des **données synthétiques** : ils n'ont pas besoin de DVC ni de Drive.

In [ ]:
# 1. Cloner le dépôt (dépôt privé : créer un token GitHub et le mettre dans les Secrets Colab sous GITHUB_TOKEN)
from google.colab import userdata
GITHUB_USER = "VOTRE-COMPTE"          # <-- à adapter
REPO = "projet-immobilier"
BRANCH = "main"                        # branche à tester

try:
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"
except Exception:                      # dépôt public
    url = f"https://github.com/{GITHUB_USER}/{REPO}.git"

!rm -rf {REPO}
!git clone --branch {BRANCH} {url} {REPO}
%cd {REPO}

In [ ]:
# 2. Dépendances minimales pour les tests
!pip install -q -r requirements-test.txt ruff==0.5.5

In [ ]:
# 3. Tests unitaires (data, modèle, API) avec couverture
!python -m pytest tests/ -v --cov=src --cov-report=term-missing

In [ ]:
# 4. Lint / format (comme la CI GitHub)
!ruff check src tests
!ruff format --check src tests

## Optionnel : tester avec le vrai modèle versionné sur Drive

Prérequis : dans les **Secrets Colab** (icône clé, à gauche) ajouter
`GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET` et `GDRIVE_CREDENTIALS_DATA`
(contenu du fichier d'identifiants OAuth généré par un premier `dvc push` en local,
voir `docs/dvc-gdrive-colab-modal.md`).

In [ ]:
!pip install -q "dvc>=3.51.2,<4" "dvc-gdrive>=3.0.1"
import os
from google.colab import userdata
for k in ["GDRIVE_CLIENT_ID", "GDRIVE_CLIENT_SECRET", "GDRIVE_CREDENTIALS_DATA"]:
    os.environ[k] = userdata.get(k)

!dvc remote modify --local storage gdrive_client_id $GDRIVE_CLIENT_ID
!dvc remote modify --local storage gdrive_client_secret $GDRIVE_CLIENT_SECRET
!dvc pull models/model.pkl models/preprocessor.pkl
!ls -lh models/

In [ ]:
# Test de l'API avec le vrai modèle (sans lancer de serveur)
from fastapi.testclient import TestClient
from src.api.main import app

client = TestClient(app)
print(client.get("/health").json())
print(client.post("/predict", json={
    "surface_reelle_bati": 65, "type_local": "Appartement",
    "code_postal": "75015", "nombre_pieces_principales": 3,
}).json())